In [1]:
pip install langdetect

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:

!pip install Sastrawi



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
pip install emoji

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [5]:
df = pd.read_csv('dataset/stock_market.csv')
df.head(10)

,Tweet Date,Sentence,Quote Count,Reply Count,Retweet Count,Favorite Count,Sentiment,English Translation
0,Thu Feb 29 11:21:27 +0000 2024,"Gk muluk muluk, 100,000 lot saham BBCA aja",0,0,0,0,Positive,"Not too ambitious, just 100,000 lots of BBCA s..."
1,Thu Feb 29 10:11:05 +0000 2024,BCA Expoversary 2024 menawarkan promo suku bun...,0,0,0,0,Neutral,BCA Expoversary 2024 offers special interest r...
2,Thu Feb 29 10:06:04 +0000 2024,[USERNAME] saham bca nya menyusul ya 🙂,0,0,0,0,Positive,[USERNAME] BCA shares will follow 🙂
3,Thu Feb 29 07:42:09 +0000 2024,PT Bank BCA Syariah (BCA Syariah) turut memeri...,0,0,0,0,Neutral,PT Bank BCA Syariah (BCA Syariah) also enliven...
4,Thu Feb 29 06:06:17 +0000 2024,[USERNAME] Begitu byk saham kamu memilih saham...,0,0,0,1,Positive,[USERNAME] So many stocks you choose those sto...
5,Thu Feb 29 05:29:15 +0000 2024,"[USERNAME] Drpd bunga mending hal² yg berguna,...",0,0,0,0,Positive,"[USERNAME] Rather than flowers, it's better to..."
6,Thu Feb 29 04:08:37 +0000 2024,[USERNAME] [USERNAME] Pilihan RDN di Mirae Ass...,0,0,0,0,Neutral,[USERNAME] [USERNAME] There are many RDN optio...
7,Thu Feb 29 04:02:04 +0000 2024,[USERNAME] [USERNAME] pak buat sekuritas mirae...,0,1,0,0,Neutral,"[USERNAME] [USERNAME] Sir, for Mirae securitie..."
8,Thu Feb 29 03:41:47 +0000 2024,"[USERNAME] Bbca, soalnya stabil",0,0,0,0,Positive,"[USERNAME] Bbca, because it's stable"
9,Thu Feb 29 03:34:36 +0000 2024,Menggelar pameran masih menjadi salah satu str...,0,1,0,1,Neutral,Holding exhibitions is still one of the mainst...


In [6]:
print(df.columns)

Index(['Tweet Date', 'Sentence', 'Quote Count', 'Reply Count', 'Retweet Count',
       'Favorite Count', 'Sentiment', 'English Translation'],
      dtype='object')


In [7]:
df.columns = df.columns.str.strip()

In [8]:
df.head(10)

,Tweet Date,Sentence,Quote Count,Reply Count,Retweet Count,Favorite Count,Sentiment,English Translation
0,Thu Feb 29 11:21:27 +0000 2024,"Gk muluk muluk, 100,000 lot saham BBCA aja",0,0,0,0,Positive,"Not too ambitious, just 100,000 lots of BBCA s..."
1,Thu Feb 29 10:11:05 +0000 2024,BCA Expoversary 2024 menawarkan promo suku bun...,0,0,0,0,Neutral,BCA Expoversary 2024 offers special interest r...
2,Thu Feb 29 10:06:04 +0000 2024,[USERNAME] saham bca nya menyusul ya 🙂,0,0,0,0,Positive,[USERNAME] BCA shares will follow 🙂
3,Thu Feb 29 07:42:09 +0000 2024,PT Bank BCA Syariah (BCA Syariah) turut memeri...,0,0,0,0,Neutral,PT Bank BCA Syariah (BCA Syariah) also enliven...
4,Thu Feb 29 06:06:17 +0000 2024,[USERNAME] Begitu byk saham kamu memilih saham...,0,0,0,1,Positive,[USERNAME] So many stocks you choose those sto...
5,Thu Feb 29 05:29:15 +0000 2024,"[USERNAME] Drpd bunga mending hal² yg berguna,...",0,0,0,0,Positive,"[USERNAME] Rather than flowers, it's better to..."
6,Thu Feb 29 04:08:37 +0000 2024,[USERNAME] [USERNAME] Pilihan RDN di Mirae Ass...,0,0,0,0,Neutral,[USERNAME] [USERNAME] There are many RDN optio...
7,Thu Feb 29 04:02:04 +0000 2024,[USERNAME] [USERNAME] pak buat sekuritas mirae...,0,1,0,0,Neutral,"[USERNAME] [USERNAME] Sir, for Mirae securitie..."
8,Thu Feb 29 03:41:47 +0000 2024,"[USERNAME] Bbca, soalnya stabil",0,0,0,0,Positive,"[USERNAME] Bbca, because it's stable"
9,Thu Feb 29 03:34:36 +0000 2024,Menggelar pameran masih menjadi salah satu str...,0,1,0,1,Neutral,Holding exhibitions is still one of the mainst...


# Cleaning Text

In [9]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['clean_text'] = df['Sentence'].apply(clean_text)

# Stopword Bahasa Indonesia

In [10]:
factory = StopWordRemoverFactory()
stopwords = factory.get_stop_words()

def remove_stopwords(text):
    return " ".join([word for word in text.split() if word not in stopwords])

df['clean_text'] = df['clean_text'].apply(remove_stopwords)

# Stemming

In [11]:
stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

df['clean_text'] = df['clean_text'].apply(lambda x: stemmer.stem(x))

# Normalisasi Slang

In [12]:
slang_dict = {
    "gk": "tidak",
    "ga": "tidak",
    "nggak": "tidak",
    "bgt": "banget",
    "dr": "dari",
    "yg": "yang",
    "jd": "jadi",
    "krn": "karena",
    "dgn": "dengan",
    "pas" : "saat"
}

def normalize_slang(text):
    words = text.split()
    normalized = [slang_dict[word] if word in slang_dict else word for word in words]
    return " ".join(normalized)

df['clean_text'] = df['clean_text'].apply(normalize_slang)

# Remove Emoji

In [14]:
def remove_emoji_regex(text):
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols
        u"\U0001F680-\U0001F6FF"  # transport
        u"\U0001F1E0-\U0001F1FF"  # flags
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

# Encode Label

In [15]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['Sentiment'])

print(dict(zip(le.classes_, le.transform(le.classes_))))

{'Negative': 0, 'Neutral': 1, 'Positive': 2}


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'],
    df['label'],
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

# TF-IDF

In [17]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=3,
    max_df=0.9
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [18]:
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))

In [19]:
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print(classification_report(y_test, y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

    Negative       0.69      0.78      0.73       157
     Neutral       0.63      0.71      0.66       147
    Positive       0.86      0.77      0.81       354

    accuracy                           0.76       658
   macro avg       0.73      0.75      0.74       658
weighted avg       0.77      0.76      0.76       658



In [23]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

xgb_model.fit(X_train_tfidf, y_train)

y_pred_xgb = xgb_model.predict(X_test_tfidf)

print("XGBoost")
print(classification_report(y_test, y_pred_xgb, target_names=le.classes_))

XGBoost
              precision    recall  f1-score   support

    Negative       0.74      0.68      0.71       157
     Neutral       0.70      0.48      0.57       147
    Positive       0.75      0.88      0.81       354

    accuracy                           0.74       658
   macro avg       0.73      0.68      0.70       658
weighted avg       0.74      0.74      0.73       658

